# Multimodal Model — GPU Training on Google Colab

Train the transformer on Wikipedia + Gutenberg + GitHub code + LibriVox audiobooks using a Colab GPU.

**Setup (do this first):**
1. **Runtime → Change runtime type → T4 GPU**
2. Put the project in Google Drive at `My Drive/transformer/`
3. Run **Cell 2** to mount Drive and open the project

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Mount Google Drive and open the project
from google.colab import drive
from pathlib import Path
import os, sys

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/transformer')
assert (PROJECT / 'train.py').exists(), f'Put project at {PROJECT}'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(f'Working directory: {PROJECT}')

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print('Ready on', torch.cuda.get_device_name(0))"

Setup complete — continue to the download cells below.

## Download training data (run on Colab — faster than uploading 10+ GB)

Download cells fetch raw data only. Run the **Process once** cell after downloads finish.

Gutenberg uses the Zenodo subset set in the download cell (`GUTENBERG_SUBSET`, default `1.7gb` / 563 MB download). Change it there if you want a different size.

Skip downloads if raw files are already present. Skip processing if `data/processed/corpus.jsonl` and `data/tokenizer.json` exist.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from config import (
    DataConfig,
    find_wikipedia_dump,
    GUTENBERG_ZENODO_SUBSETS,
    gutenberg_zenodo_path,
    read_gutenberg_subset,
)
from utils.gutenberg_archive import count_raw_txt, remove_legacy_bulk_artifacts

PROJECT = next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists())
os.chdir(PROJECT)
cfg = DataConfig()
py = sys.executable

# Wikipedia — pages-articles (current revisions only). Small shard ~300 MB by default.
wiki_dump = find_wikipedia_dump(cfg.wiki_dump_dir)
if wiki_dump is None:
    !python download_wikipedia.py --small
else:
    print(f'Wikipedia dump already present: {wiki_dump.name}')

# Gutenberg — Zenodo subset (NOT the legacy 10 GB bulk archive)
# https://zenodo.org/records/3360392 — change subset: 184mb, 357mb, 670mb, 1gb, 1.7gb
GUTENBERG_SUBSET = '1.7gb'
gutenberg_dir = cfg.gutenberg_dir
raw_dir = gutenberg_dir / 'raw'
removed_legacy = remove_legacy_bulk_artifacts(gutenberg_dir)
if removed_legacy:
    print(f'Removed legacy bulk archive: {", ".join(removed_legacy)}')

raw_count = count_raw_txt(raw_dir)
if raw_count > 0:
    print(f'Gutenberg raw/ ready ({raw_count:,} files) — skipping download/extract')
else:
    help_text = subprocess.run(
        [py, 'download_gutenberg.py', '--help'],
        capture_output=True, text=True, check=True,
    ).stdout
    if 'zenodo' not in help_text:
        raise RuntimeError(
            'Stale download_gutenberg.py on Drive still uses the old bulk downloader. '
            'Sync this project to Drive, restart the Colab runtime, then re-run from Cell 2.'
        )
    zip_path = gutenberg_zenodo_path(gutenberg_dir, GUTENBERG_SUBSET)
    subset_meta = GUTENBERG_ZENODO_SUBSETS[GUTENBERG_SUBSET]
    print(f"Gutenberg subset: {GUTENBERG_SUBSET} — {subset_meta['label']}")
    existing = read_gutenberg_subset(gutenberg_dir)
    if existing and existing != GUTENBERG_SUBSET:
        print(f'Note: found {existing} on disk; using {GUTENBERG_SUBSET} from this cell')
    min_bytes = int(subset_meta['bytes']) * 0.99
    if not zip_path.exists() or zip_path.stat().st_size < min_bytes:
        subprocess.run(
            [py, 'download_gutenberg.py', '--method', 'zenodo', '--subset', GUTENBERG_SUBSET],
            check=True,
        )
    else:
        print(f'Gutenberg zip already present: {zip_path.name}')
    subprocess.run([py, 'extract_gutenberg.py', '--subset', GUTENBERG_SUBSET], check=True)

# GitHub — Python/JS/TS/Kotlin/Swift repos (set GITHUB_TOKEN in Colab secrets for faster downloads)
github_raw = cfg.github_dir / 'raw'
if not github_raw.is_dir() or not list(github_raw.glob('*.zip')):
    if not os.environ.get('GITHUB_TOKEN'):
        print('Tip: add GITHUB_TOKEN in Colab secrets for higher rate limits')
    !python download_github.py --max-repos 50
else:
    print(f'GitHub raw data present ({len(list(github_raw.glob("*.zip")))} repos)')

In [ ]:
# LibriVox — English audiobooks (download only; Whisper runs in Process once cell)
import os
from pathlib import Path
from config import DataConfig

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
cfg = DataConfig()
if not (cfg.librivox_dir / 'books.jsonl').exists():
    !python download_librivox.py --max-books 25
else:
    print('LibriVox manifest already present')

In [ ]:
# Process once — per-source JSONL, merge, wiki QA, tokenizer (skips if already built)
import subprocess
import sys
import os
from pathlib import Path
from config import DataConfig

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
cfg = DataConfig()

!python check_data_ready.py || true

if cfg.corpus_path.exists() and cfg.tokenizer_path.exists():
    print('Corpus and tokenizer already built — skipping processing')
else:
    cmd = [
        sys.executable, 'process_all.py',
        '--whisper-device', 'cuda',
        '--whisper-model', 'base',
        '--max-librivox-books', '25',
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(
            f'process_all.py failed (exit {result.returncode}). '
            'Run check_data_ready.py and ensure download cells finished first.'
        )

!python check_data_ready.py

## GPU Training — pick one cell below

- **Colab Pro (A100/V100/high-RAM):** run the Pro cell — full model, larger batches, more QA eval
- **Free T4 / OOM issues:** run the Safe cell — smaller model, low memory

In [ ]:
# Colab PRO — full model, faster training (A100 / V100 / high-RAM T4)
import os
import torch
from pathlib import Path

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
print(f'Working directory: {os.getcwd()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!python check_data_ready.py
!python train.py --device cuda --epochs 3 --batch-size 8 --grad-accum-steps 4 --max-seq-len 1024 --val-batch-size 4 --eval-every 100 --qa-samples 50 --num-workers 2

In [ ]:
# SAFE — free T4 or if Pro cell hits OOM
import os
from pathlib import Path

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
print(f'Working directory: {os.getcwd()}')
!python check_data_ready.py
!python train.py --device cuda --small --epochs 3 --batch-size 2 --grad-accum-steps 8 --max-seq-len 512 --val-batch-size 2 --eval-every 200 --qa-samples 10 --num-workers 0

In [ ]:
# Evaluate best checkpoint
import os
from pathlib import Path

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
!python evaluate.py --checkpoint checkpoints/best.pt --qa-samples 50

In [ ]:
# Test inference
import os
from pathlib import Path

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
!python inference.py --prompt "Paris is the capital of" --checkpoint checkpoints/best.pt --max-tokens 40

In [ ]:
# Save checkpoints to Google Drive (recommended)
from google.colab import drive
import shutil
from pathlib import Path
import os
from pathlib import Path

os.chdir(next(p for p in [Path('/content/transformer'), Path('/content'), Path('/content/drive/MyDrive/transformer')] if (p / 'train.py').exists()))
drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/transformer_checkpoints')
dest.mkdir(parents=True, exist_ok=True)
for f in Path('checkpoints').glob('*.pt'):
    shutil.copy(f, dest / f.name)
shutil.copy('data/tokenizer.json', dest / 'tokenizer.json')
print(f'Saved to {dest}')